In [1]:
import h5py 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
def get_target_percent(filename_path, limits=[-75, 75]):
    df = pd.read_csv(filename_path)
    df["error"] = pd.to_numeric(df["error"], errors="coerce")

    if "is_sst_trial" in df:
        marker = df["is_sst_trial"].astype(str).str.strip().str.lower()
        stop_mask = marker.isin({"1", "true", "yes", "y"})
    elif "stimulus_filename" in df:
        stop_mask = df["stimulus_filename"].astype(str).str.lower().str.contains("stop", na=False)
    else:
        stop_mask = pd.Series(False, index=df.index)

    is_sst_file = bool(stop_mask.any())
    target_df = df.loc[~stop_mask].copy() if is_sst_file else df

    n_target = target_df.loc[(target_df.error > limits[0]) & (target_df.error < limits[1])].shape[0]
    p_target = np.nan if target_df.shape[0] == 0 else (n_target / target_df.shape[0]) * 100
    label = "go stimuli without stop" if is_sst_file else "all stimuli"
    print(f"Target percent in {limits} ({label}) - {p_target:.1f} % ({n_target}/{target_df.shape[0]})")
    print(f"mean +- std - {np.nanmean(target_df.error):.0f}+-{np.nanstd(target_df.error):.0f}")
    print(f"median - {np.nanmedian(target_df.error):.0f}")

    if is_sst_file:
        stop_df = df.loc[stop_mask].copy()
        n_sst_errors = int(stop_df.error.notna().sum())
        p_sst_errors = np.nan if stop_df.shape[0] == 0 else (n_sst_errors / stop_df.shape[0]) * 100
        print(f"SST error percent - {p_sst_errors:.1f} % ({n_sst_errors}/{stop_df.shape[0]})")

In [3]:
subject = "08EN_day3"

In [4]:
records = os.listdir(os.path.join(r"../data", subject))
records = [record for record in records if (record.find("hdf") == -1) & (record.find("tms") == -1) & (record.find("png") == -1)\
           &(record.find("txt") == -1)&(record.find("asc") == -1)]
records

['01_08EN_day3_efb.csv',
 '02_08EN_day3_efb.csv',
 '03_08EN_day3_nofb_test.csv',
 '04_08EN_day3_nofb_test.csv',
 '05_08EN_day3_sst_training.csv',
 '06_08EN_day3_fb.csv']

In [5]:
limit = 75
for record in records:
    print("--------", record, "--------")
    get_target_percent(os.path.join(r"../data", subject, record), limits=[-limit, limit])


-------- 01_08EN_day3_efb.csv --------
Target percent in [-75, 75] (all stimuli) - 70.0 % (28/40)
mean +- std - -23+-78
median - -7
-------- 02_08EN_day3_efb.csv --------
Target percent in [-75, 75] (all stimuli) - 80.0 % (32/40)
mean +- std - -43+-122
median - -25
-------- 03_08EN_day3_nofb_test.csv --------
Target percent in [-75, 75] (all stimuli) - 58.6 % (17/29)
mean +- std - 52+-66
median - 51
-------- 04_08EN_day3_nofb_test.csv --------
Target percent in [-75, 75] (all stimuli) - 56.7 % (17/30)
mean +- std - -53+-57
median - -41
-------- 05_08EN_day3_sst_training.csv --------
Target percent in [-75, 75] (go stimuli without stop) - 73.3 % (22/30)
mean +- std - -55+-48
median - -56
SST error percent - 46.7 % (7/15)
-------- 06_08EN_day3_fb.csv --------
Target percent in [-75, 75] (all stimuli) - 63.3 % (19/30)
mean +- std - -6+-108
median - 14


In [8]:
np.asarray([169, 358, 440, 517, 738, 816, 895, 980]) + 74

array([ 243,  432,  514,  591,  812,  890,  969, 1054])

In [9]:
np.asarray([139, 601, 1057]) + 79

array([ 218,  680, 1136])

In [20]:
630-79

551